# MA2006B: Curvas Elípticas sobre 𝔽_p y ECDH

## Parte 2 — Curvas Elípticas sobre el Campo Primo 𝔽_p

Implementación de `EllipticCurveFp` y `simulate_ecdh` según las instrucciones del proyecto.

In [ ]:
class EllipticCurveFp:
    """
    Curva elíptica sobre el campo primo 𝔽_p.

    La curva sigue la forma de Weierstrass corta:
        E : y^2 = x^3 + a x + b 

    El punto al infinito se representa como (None, None).
    """

    def __init__(self, a: int, b: int, p: int) -> None:
        if p <= 2:
            raise ValueError('p must be greater than 2')
        self.p = p
        self.a = a % p
        self.b = b % p

        delta = (4 * pow(a, 3, p) + 27 * pow(b, 2, p)) % p
        if delta == 0:
            raise ValueError('Curve is singular modulo p')

    def mod_inverse(self, k: int, p: int) -> int:
        k = k % p
        if k == 0:
            raise ZeroDivisionError('No inverse exists')

        a, b = k, p
        x0, x1 = 1, 0
        while b != 0:
            q = a // b
            a, b = b, a - q * b
            x0, x1 = x1, x0 - q * x1

        if a != 1:
            raise ZeroDivisionError('No inverse exists')

        inverse = x0 % p
        if inverse == 0:
            inverse = p
        return inverse

    def is_on_curve(self, P: tuple[int, int] | tuple[None, None]) -> bool:
        if P == (None, None):
            return True

        x, y = P
        left = (y * y) % self.p
        right = (x * x * x + self.a * x + self.b) % self.p
        return left == right

    def add_points(self, P: tuple, Q: tuple) -> tuple:
        if not self.is_on_curve(P):
            raise ValueError('Point not on curve')
        if not self.is_on_curve(Q):
            raise ValueError('Point not on curve')

        if P == (None, None):
            return Q
        if Q == (None, None):
            return P

        x1, y1 = P
        x2, y2 = Q

        if x1 == x2 and (y1 + y2) % self.p == 0:
            return (None, None)

        if x1 == x2 and y1 == y2:
            if y1 % self.p == 0:
                return (None, None)
            m = ((3 * x1 * x1 + self.a) * self.mod_inverse(2 * y1, self.p)) % self.p
        else:
            m = ((y2 - y1) * self.mod_inverse(x2 - x1, self.p)) % self.p

        x3 = (m * m - x1 - x2) % self.p
        y3 = (m * (x1 - x3) - y1) % self.p
        return (x3, y3)

    def scalar_multiply(self, k: int, P: tuple) -> tuple:
        if not self.is_on_curve(P):
            raise ValueError('Point not on curve')

        if k == 0:
            return (None, None)

        if k < 0:
            x, y = P
            return self.scalar_multiply(-k, (x, (-y) % self.p))

        result = (None, None)
        addend = P

        while k > 0:
            if k & 1:
                result = self.add_points(result, addend)
            addend = self.add_points(addend, addend)
            k >>= 1

        return result

In [ ]:
def simulate_ecdh(curve: EllipticCurveFp, G: tuple, d_A: int, d_B: int) -> tuple:
    Q_A = curve.scalar_multiply(d_A, G)
    Q_B = curve.scalar_multiply(d_B, G)

    S_A = curve.scalar_multiply(d_A, Q_B)
    S_B = curve.scalar_multiply(d_B, Q_A)

    assert S_A == S_B
    return S_A

In [ ]:
curve_fp = EllipticCurveFp(a=2, b=2, p=17)

assert curve_fp.is_on_curve((5, 1)) == True
assert curve_fp.add_points((5, 1), (5, 1)) == (6, 3)
assert curve_fp.add_points((5, 1), (10, 6)) == (3, 1)
assert curve_fp.scalar_multiply(10, (5, 1)) == (7, 11)
assert curve_fp.scalar_multiply(19, (5, 1)) == (None, None)
assert simulate_ecdh(curve_fp, (5, 1), 3, 7) == (6, 3)

print('Parte 2 OK: EllipticCurveFp y ECDH funcionan correctamente')